**Imports**

In [13]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Subset,ConcatDataset
from torchvision import datasets,transforms
from sklearn.metrics import roc_auc_score

**OE Loss**

In [14]:
def outlier_exposure_loss(logits_oe):
    log_prob = nn.LogSoftmax(dim=1)(logits_oe)
    return -log_prob.mean()

**Data**

In [15]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

import os
data_dir = os.path.expanduser('~/.pytorch/data') 

mnist_train  = datasets.MNIST(root=data_dir, train=True,  download=True, transform=transform)
mnist_test    = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)
fashion_train = datasets.FashionMNIST(root=data_dir, train=True,  download=True, transform=transform)
fashion_test  = datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=transform)

id_labels = set(range(1, 10))
label_map = {orig: i for i, orig in enumerate(sorted(id_labels))}

def filter_indices(dataset, labels_to_keep):
    return [i for i, (_, y) in enumerate(dataset) if y in labels_to_keep]

class RemappedSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, label_map):
        self.dataset, self.indices, self.label_map = dataset, indices, label_map
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        return x, self.label_map[y]

id_train_idx = filter_indices(mnist_train, id_labels)
id_train_set = RemappedSubset(mnist_train, id_train_idx, label_map)

oe_idx = torch.randperm(len(fashion_train))[:len(id_train_set)].tolist()
oe_set = Subset(fashion_train, oe_idx)

id_test_idx  = filter_indices(mnist_test, id_labels)
id_test_set  = RemappedSubset(mnist_test, id_test_idx, label_map)
zero_idx     = filter_indices(mnist_test, {0})

id_test_loader   = DataLoader(id_test_set, batch_size=256)
zero_loader      = DataLoader(Subset(mnist_test, zero_idx), batch_size=256)
fashion_loader   = DataLoader(fashion_test, batch_size=256) 
noise_dataset    = torch.randn(2000, 1, 28, 28)
noise_loader     = DataLoader(noise_dataset, batch_size=256)

**Model Architecture**

In [16]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.conv1   = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2   = nn.Conv2d(32, 64, 3, padding=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.relu    = nn.ReLU()
        self.fc1     = nn.Linear(64 * 7 * 7, 128)
        self.fc2     = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(self.dropout(x))

**Training with OE**

In [17]:
device = "mps" if torch.backends.mps.is_available else "cpu"

def train_epoch_oe(model,id_loader,oe_loader,optimizer,lmbd=0.5):
    model.train()
    
    total_loss,correct,total = 0,0,0
    oe_iter = iter(oe_loader)
    
    for X_in,y_in in id_loader:
        X_in,y_in = X_in.to(device),y_in.to(device)
        try:
            X_oe,y_oe = next(oe_iter)
        except StopIteration:
            oe_iter = iter(oe_loader)
            X_oe,y_oe = next(oe_iter)
        X_oe = X_oe.to(device)
        
        optimizer.zero_grad()
        
        logits_in = model(X_in)
        loss_ce = nn.CrossEntropyLoss()(logits_in,y_in)
        
        logits_oe = model(X_oe)
        loss_oe = outlier_exposure_loss(logits_oe)
        
        loss = loss_ce + lmbd*loss_oe
        loss.backward()
        optimizer.step()
        
        total_loss+=loss.item()*X_in.shape[0]
        correct+=(logits_in.argmax(1)==y_in).sum().item()
        total+=X_in.shape[0]
        
    return total_loss/total,correct/total
        
@torch.no_grad()
def evaluate_accuracy(model,loader):
    model.eval()
    correct,total = 0,0
    
    for X,y in loader:
        X,y = X.to(device),y.to(device)
        correct = (model(X).argmax(1)==y).sum().item()
        total+=X.shape[0]
    
    return correct/total

**OOD Scoring**

In [18]:
@torch.no_grad()
def msp_scores(model,loader,is_labelled=True):
    model.eval()
    scores = []
    
    for batch in loader:
        X = batch[0] if is_labelled else batch
        X = X.to(device)
        probs = nn.Softmax(dim=1)(model(X))
        scores.append(probs.max(dim=1).values.cpu())
    return torch.cat(scores)

def auroc(id_scores,ood_scores):
    y_true = [0]*len(id_scores) + [1]*len(ood_scores)    
    y_score = torch.cat([-id_scores,-ood_scores]).numpy()
    return roc_auc_score(y_true,y_score)

**Train**

In [19]:
id_loader = DataLoader(id_train_set,batch_size=128,shuffle=True)
oe_loader = DataLoader(oe_set,batch_size=128,shuffle=True)

model_oe = SmallCNN(num_classes=9).to(device)
optimizer = torch.optim.Adam(model_oe.parameters(),lr=1e-3)

EPOCHS = 10
for epoch in range(EPOCHS):
    loss,acc = train_epoch_oe(model_oe,id_loader,oe_loader,optimizer,lmbd=0.5)
    id_acc = evaluate_accuracy(model_oe,id_test_loader)
    print(f"Epoch {epoch+1}: loss={loss:.4f}, train_acc={acc:.4f}, id_test_acc={id_acc:.4f}")

Epoch 1: loss=1.3044, train_acc=0.9411, id_test_acc=0.0067
Epoch 2: loss=1.1589, train_acc=0.9821, id_test_acc=0.0065
Epoch 3: loss=1.1442, train_acc=0.9865, id_test_acc=0.0065
Epoch 4: loss=1.1331, train_acc=0.9899, id_test_acc=0.0067
Epoch 5: loss=1.1273, train_acc=0.9910, id_test_acc=0.0067
Epoch 6: loss=1.1231, train_acc=0.9926, id_test_acc=0.0067
Epoch 7: loss=1.1187, train_acc=0.9938, id_test_acc=0.0067
Epoch 8: loss=1.1171, train_acc=0.9940, id_test_acc=0.0067
Epoch 9: loss=1.1143, train_acc=0.9951, id_test_acc=0.0067
Epoch 10: loss=1.1124, train_acc=0.9957, id_test_acc=0.0067


**OOD Detection Evaluation**

In [20]:
id_s      = msp_scores(model_oe,id_test_loader)
zero_s    = msp_scores(model_oe,zero_loader)
fashion_s = msp_scores(model_oe,fashion_loader)
noise_s   = msp_scores(model_oe,noise_loader,is_labelled=False)

print(f"\nOE + MSP Results:")
print(f"  AUROC vs digit 0:      {auroc(id_s,zero_s):.4f}")
print(f"  AUROC vs FashionMNIST: {auroc(id_s,fashion_s):.4f}")
print(f"  AUROC vs noise:        {auroc(id_s,noise_s):.4f}")


OE + MSP Results:
  AUROC vs digit 0:      0.9792
  AUROC vs FashionMNIST: 1.0000
  AUROC vs noise:        1.0000


**Analysis**

Perfect Detection for Noise and FashionMNIST as OOD Samples

Very good performance for digit 0 as well, which means exposure to Outlier samples is very generalisable - Exposing the model to auxillary outlier datasets makes the understanding of model for ID much stricter and it stops lazily classifying as per some curves in sample (example 0's curve would no longer be easily confused with 8,9,etc)